# Notebook 4: Analyze Xenium Data

**Source:** [Squidpy Tutorial — Xenium](https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_xenium.html)

This notebook demonstrates how to analyze **10x Genomics Xenium** data — a sub-cellular resolution, single-molecule spatial transcriptomics platform. Unlike Visium (which aggregates transcripts per tissue spot), Xenium resolves transcripts at the **single-cell level**.

**Dataset:** Xenium Human Lung Cancer — 161,000 cells × 480 genes  
**Framework:** SpatialData + Squidpy

## Topics Covered
1. Reading Xenium data with `spatialdata-io` and converting to Zarr
2. Quality control metrics (transcripts, cell area, nucleus ratio)
3. Normalization, PCA, UMAP, Leiden clustering
4. Spatial visualization of clusters
5. Centrality scores (closeness, degree, clustering coefficient)
6. Ripley statistics for spatial point pattern analysis

---
> **Data Download Required**  
> Download the [Xenium Human Lung Cancer dataset](https://www.10xgenomics.com/datasets/preview-data-ffpe-human-lung-cancer-with-xenium-multimodal-cell-segmentation-1-standard) from 10x Genomics.  
> Extract it into a directory named `Xenium/` in the project root (`../Xenium/`).

## 0. Import Libraries

In [ ]:
import spatialdata as sd
from spatialdata_io import xenium

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import squidpy as sq

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")
print(f"spatialdata=={sd.__version__}")

## 1. Load Xenium Data

The `spatialdata-io` library provides a reader for Xenium output directories.  
We convert the data to **Zarr** format for efficient on-disk access.

In [ ]:
# Paths — adjust if your data is in a different location
xenium_path = "../Xenium"          # raw Xenium output directory
zarr_path   = "../Xenium.zarr"     # converted Zarr store

import os

if not os.path.exists(xenium_path):
    raise FileNotFoundError(
        f"Xenium data not found at '{xenium_path}'.\n"
        "Please download the dataset from 10x Genomics and extract it there."
    )

# Parse Xenium output into a SpatialData object
sdata = xenium(xenium_path)

# Convert to Zarr for efficient storage and access
sdata.write(zarr_path)
print("SpatialData written to Zarr successfully.")

In [ ]:
# Load directly from Zarr from now on
sdata = sd.read_zarr(zarr_path)
print(sdata)

The `SpatialData` object contains:
- **Images**: high-resolution morphology image (multiscale)
- **Labels**: cell and nucleus segmentation masks
- **Points**: individual transcript locations (~40M transcripts)
- **Shapes**: cell and nucleus boundary polygons
- **Tables**: AnnData with per-cell count matrix

In [ ]:
# Extract AnnData (count matrix + metadata)
adata = sdata.tables["table"]
print(adata)
print("\nFirst 5 rows of cell metadata:")
adata.obs.head()

The `obs` table includes per-cell metadata:
- `transcript_counts`: number of detected transcripts
- `control_probe_counts` / `control_codeword_counts`: negative controls for QC
- `cell_area` / `nucleus_area`: segmented cell and nucleus sizes

## 2. Quality Control

Xenium-specific QC includes:
- **Control probe %** — measures background signal (should be very low)
- **Control codeword %** — measures decoding errors
- **Total transcripts per cell** — cells with very few transcripts may be debris
- **Cell area** — unusually large/small cells may be segmentation artifacts
- **Nucleus ratio** — nucleus area / cell area (should be biologically sensible)

In [ ]:
# Calculate standard QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    percent_top=(10, 20, 50, 150),
    inplace=True
)

In [ ]:
# Xenium-specific control metrics
cprobes = (
    adata.obs["control_probe_counts"].sum()
    / adata.obs["total_counts"].sum() * 100
)
cwords = (
    adata.obs["control_codeword_counts"].sum()
    / adata.obs["total_counts"].sum() * 100
)
print(f"Negative DNA probe count %  : {cprobes:.4f}%")
print(f"Negative decoding count %   : {cwords:.4f}%")
print("(Both should be < 1% for good quality data)")

In [ ]:
# QC distribution plots
fig, axs = plt.subplots(1, 4, figsize=(16, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])

axs[1].set_title("Unique transcripts per cell")
sns.histplot(adata.obs["n_genes_by_counts"], kde=False, ax=axs[1])

axs[2].set_title("Area of segmented cells (µm²)")
sns.histplot(adata.obs["cell_area"], kde=False, ax=axs[2])

axs[3].set_title("Nucleus / Cell area ratio")
sns.histplot(
    adata.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False,
    ax=axs[3]
)

plt.tight_layout()
plt.savefig("../figures/04_xenium_qc.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Filtering, Normalization & Preprocessing

We filter low-quality cells and genes, then follow the standard Scanpy preprocessing pipeline.

In [ ]:
# Filter cells and genes
print(f"Before filtering: {adata.n_obs} cells, {adata.n_vars} genes")

sc.pp.filter_cells(adata, min_counts=10)   # remove cells with fewer than 10 transcripts
sc.pp.filter_genes(adata, min_cells=5)     # remove genes expressed in fewer than 5 cells

print(f"After filtering:  {adata.n_obs} cells, {adata.n_vars} genes")

In [ ]:
# Save raw counts for later use (e.g. differential expression on raw counts)
adata.layers["counts"] = adata.X.copy()

# Normalize to total counts per cell, then log-transform
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)

# Dimensionality reduction and clustering
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata)

print(f"Found {adata.obs['leiden'].nunique()} Leiden clusters")

## 4. Visualization

We visualize the Leiden clusters in both **UMAP** (transcriptional similarity) and **spatial** (tissue context) coordinates.

In [ ]:
# UMAP — color by QC metrics and cluster
sc.pl.umap(
    adata,
    color=["total_counts", "n_genes_by_counts", "leiden"],
    wspace=0.4
)

In [ ]:
# Spatial scatter — single-cell resolution, colored by Leiden cluster
# shape=None: plot cells as points (no spot circles like Visium)
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None,
    color=["leiden"],
    wspace=0.4
)

## 5. Spatial Statistics — Centrality Scores

Centrality scores describe the spatial organization of each cluster **within the tissue graph**.

| Score | Definition |
|-------|------------|
| **Closeness centrality** | How close a cluster's cells are to all other cells — measures spatial compactness |
| **Degree centrality** | Fraction of non-cluster cells directly connected to cluster cells |
| **Clustering coefficient** | How densely interconnected the cluster's neighbors are — measures local clustering |

These are all derived from the spatial neighborhood graph.

In [ ]:
# Build spatial neighborhood graph
# coord_type='generic' is appropriate for Xenium (not a regular Visium grid)
# n_neighs=10: each cell connected to its 10 nearest spatial neighbors
sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=10)

print("Spatial graph built.")
print(f"Connectivity matrix shape: {adata.obsp['spatial_connectivities'].shape}")

In [ ]:
# Compute centrality scores per Leiden cluster
sq.gr.centrality_scores(adata, cluster_key="leiden")

# Visualize all three centrality metrics
sq.pl.centrality_scores(
    adata,
    cluster_key="leiden",
    figsize=(16, 5)
)

**How to interpret centrality scores:**
- Clusters with **high closeness centrality** are spatially central in the tissue (surrounded by many cell types)
- Clusters with **high degree centrality** interface with many different cell types
- Clusters with **high clustering coefficient** form tightly packed homogeneous regions

## 6. Ripley Statistics

**Ripley's statistics** are classical spatial point process metrics that measure whether a cluster's cells are:
- **Clustered** (cells clump together more than random)
- **Random** (uniform spatial distribution)
- **Dispersed** (cells are more spread out than random)

Three variants are available:
- **Ripley's L** — normalized K function, positive values = clustering
- **Ripley's K** — counts expected vs. observed neighbors at radius r
- **Ripley's G** — nearest-neighbor distance distribution

In [ ]:
# Compute Ripley's L statistic for all clusters
# mode can be 'L', 'K', or 'G'
sq.gr.ripley(adata, cluster_key="leiden", mode="L")

# Visualize Ripley's L curves per cluster
sq.pl.ripley(
    adata,
    cluster_key="leiden",
    mode="L"
)

**Reading the Ripley's L plot:**
- The **x-axis** is the radius (distance from each cell)
- The **y-axis** is the L statistic value
- **L > 0** at a given radius → cells of that cluster are **more clustered** than a random distribution at that scale
- The peak of the curve indicates the characteristic **spatial scale** of clustering for each cluster

## 7. Neighborhood Enrichment on Xenium

Just like with Visium, we can ask which Leiden clusters preferentially co-localize in space.

In [ ]:
# Compute neighborhood enrichment
sq.gr.nhood_enrichment(adata, cluster_key="leiden")

# Visualize
sq.pl.nhood_enrichment(
    adata,
    cluster_key="leiden",
    figsize=(8, 8),
    title="Neighborhood Enrichment — Xenium Lung Cancer"
)

## 8. (Optional) Interactive Visualization with napari-spatialdata

For interactive exploration of the full-resolution Xenium dataset, you can use **napari-spatialdata**. This requires a local GUI environment (not available in headless/remote servers).

```python
# Run this in a local environment with a display
import napari
from napari_spatialdata import Interactive

interactive = Interactive(sdata)
interactive.run()
```

napari-spatialdata allows you to:
- Pan and zoom the morphology image
- Toggle cell segmentation overlays
- Color cells by any `adata.obs` column
- Inspect individual transcript locations

## Summary

This notebook covered the complete Xenium analysis workflow:

| Step | Tool / Function | Output |
|------|----------------|--------|
| Read Xenium data | `spatialdata_io.xenium()` | SpatialData object |
| Convert to Zarr | `sdata.write()` | Efficient on-disk storage |
| QC metrics | `sc.pp.calculate_qc_metrics()` | Per-cell quality stats |
| Filter + normalize | `sc.pp.filter_cells/genes()`, `normalize_total()` | Clean count matrix |
| Clustering | PCA → neighbors → UMAP → Leiden | Cell type groups |
| Spatial graph | `sq.gr.spatial_neighbors()` | Connectivity matrix |
| Centrality scores | `sq.gr.centrality_scores()` | Spatial organization metrics |
| Ripley statistics | `sq.gr.ripley()` | Spatial clustering patterns |
| Neighborhood enrichment | `sq.gr.nhood_enrichment()` | Cluster co-localization |

### Xenium vs. Visium — Key Differences

| Feature | Visium | Xenium |
|---------|--------|--------|
| Resolution | ~55 µm spots (multi-cell) | Single-cell |
| Gene panel | Whole transcriptome | Targeted panel (~400–500 genes) |
| Data format | H5 + images | SpatialData (Zarr) |
| Cell count | ~3,000–5,000 spots | 50,000–200,000+ cells |
| Spatial unit | Spot coordinates | Cell centroid coordinates |